In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

# LOAD FRESH
df = pd.read_csv("E:/programs/Projects/Anime-Recommender/data/anime.csv")
print("Original shape:", df.shape)

# CLEAN
df = df.dropna(subset=['genre'])
df['type'] = df['type'].fillna("Unknown")
df = df.dropna(subset=['rating'])
df['name'] = df['name'].str.replace('&#039;', "'", regex=False)
df['name'] = df['name'].str.replace('&amp;', '&', regex=False)
df['episodes'] = df['episodes'].replace('Unknown', 'Ongoing')
print("After cleaning:", df.shape)

# SPLIT GENRE
df['genre'] = df['genre'].str.split(", ")
print("Sample genre:", df['genre'].iloc[0])
print("Type:", type(df['genre'].iloc[0]))

# MULTI LABEL BINARIZER
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre'])
print("Genre matrix shape:", genre_matrix.shape)
print("Total unique genres:", len(mlb.classes_))
# After all cleaning steps
df = df.reset_index(drop=True)
print("Shape:", df.shape)
print("First indices:", df.index[:5].tolist())

Original shape: (12294, 7)
After cleaning: (12017, 7)
Sample genre: ['Drama', 'Romance', 'School', 'Supernatural']
Type: <class 'list'>
Genre matrix shape: (12017, 43)
Total unique genres: 43
Shape: (12017, 7)
First indices: [0, 1, 2, 3, 4]


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

rating_scaled = scaler.fit_transform(df[['rating']])
members_scaled = scaler.fit_transform(df[['members']])

combined_matrix = np.hstack([
    genre_matrix,          
    rating_scaled*2 ,
    members_scaled*1 
])

return combined_matrix


(12017, 45)


In [19]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_anime(anime_title, df, combined_matrix, n=10):
    try:
        idx = df[df['name'] == anime_title].index[0]
    except:
        print(f"{anime_title} not found!")
        return None
    
    anime_vector = combined_matrix[idx].reshape(1, -1)
    sim_score = cosine_similarity(anime_vector, combined_matrix)[0]
    sim_score[idx] = -1
    sim_indices = np.argsort(sim_score)[::-1][:n*3]  # get more first
    
    result = df[['name', 'genre', 'type', 'rating',
                 'members', 'episodes']].iloc[sim_indices].copy()
    result['similarity'] = sim_score[sim_indices]
    
    # Filter by minimum members
    result = result[result['members'] >= 800]
    
    return result.head(n)

print(recommend_anime("Steins;Gate", df, combined_matrix))

                                                   name               genre  \
59           Steins;Gate Movie: Fuka Ryouiki no Déjà vu  [Sci-Fi, Thriller]   
126               Steins;Gate: Oukoubakko no Poriomania  [Sci-Fi, Thriller]   
196   Steins;Gate: Kyoukaimenjou no Missing Link - D...  [Sci-Fi, Thriller]   
1578           Sakasama no Patema: Beginning of the Day            [Sci-Fi]   
1594                                 Mai-Otome 0: S.ifr            [Sci-Fi]   
3491                      Glass no Hana to Kowasu Sekai            [Sci-Fi]   
5281                  Final Fantasy: The Spirits Within            [Sci-Fi]   
3579                                  Fireball Charming            [Sci-Fi]   
4149                                          Je T'aime            [Sci-Fi]   
4892  Computer Kakumei: Saikyou x Saisoku no Zunou T...            [Sci-Fi]   

         type  rating  members episodes  similarity  
59      Movie    8.61   192424        1    0.980207  
126   Special    8.46 